<center>
<img src="https://laelgelcpublic.s3.sa-east-1.amazonaws.com/lael_50_years_narrow_white.png.no_years.400px_96dpi.png" width="300" alt="LAEL 50 years logo">
<h3>APPLIED LINGUISTICS GRADUATE PROGRAMME (LAEL)</h3>
</center>
<hr>

# Corpus Linguistics - Study 1 - Phase 2 - Tatiana

## Load image source data 1

In [1]:
import pandas as pd

df_images_source_1 = pd.read_csv(
    "docs/text_uniq_first_with_photo_view.tsv",
    sep="\t"
)

## Load image source data 2

In [2]:
import pandas as pd

df_images_source_2 = pd.read_csv(
    "docs/photos_url_view.tsv",
    sep="\t"
)

## Load image source data 3

In [3]:
import pandas as pd

df_images_source_3 = pd.read_csv(
    "docs/tati_oms/tati_scores_only.tsv",
    sep="\t"
)

## Filter tweets with image-source matches

In [4]:
df_tweets = df_images_source_1[
    df_images_source_1["text_uniq"].isin(df_images_source_3["file"])
].copy()

Drop the `text_emojified` column

In [5]:
df_tweets = df_tweets.drop(columns=["text_emojified"])

## Copy selected Tatiana images

In [6]:
from pathlib import Path
import shutil

source_dir = Path("corpus/cl_st1_ph2_tatiana_images_all")
target_dir = Path("corpus/cl_st1_ph2_tatiana_images")
target_dir.mkdir(parents=True, exist_ok=True)

photo_ids = (
    df_tweets["photo_uniq_id"]
    .dropna()
    .astype(str)
    .unique()
)

copied = 0
missing = []

for filename in photo_ids:
    src = source_dir / filename
    dst = target_dir / filename

    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing.append(filename)

print(f"Copied {copied} files to {target_dir}")
print(f"Missing {len(missing)} files")

if missing:
    print("First missing files:")
    print("\n".join(missing[:20]))

Copied 14450 files to corpus/cl_st1_ph2_tatiana_images
Missing 0 files


In [7]:
df_tweets

,file,created_at,text_uniq,conversation_id,author_id,username,tweet_url,text,photo_uniq_id
1,saude202003_n_00001.json,2020-03-30 23:58:27,tweet_00000006,convo_1244774019859206145,id_173533635,unknown,unknown,"Lembrando q estamos atendendo somente on-line,...",saude202003_n_00001_00000001.jpg
2,saude202003_n_00001.json,2020-03-30 23:54:45,tweet_00000007,convo_1244774019859206145,id_173533635,unknown,unknown,"Não é muito, mas é a forma que encontramos par...",saude202003_n_00001_00000001.jpg
3,saude202003_n_00001.json,2020-03-30 23:50:57,tweet_00000022,convo_1244774142701981698,id_454396343,unknown,unknown,"Segundo levantamento da AFP (dados da OMS) 3,3...",saude202003_n_00001_00000002.jpg
4,saude202003_n_00002.json,2020-03-30 23:27:18,tweet_00000051,convo_1244693362361487364,id_191068627,sf79,https://twitter.com/sf79/status/12446933623614...,@cmelo @Francischini_ @jairbolsonaro A OMS nun...,saude202003_n_00002_00000001.jpg
5,saude202003_n_00002.json,2020-03-30 23:26:49,tweet_00000054,convo_1244768070524166145,id_199501770,unknown,unknown,OMS diz que não há transmissão aérea de Covid-...,saude202003_n_00002_00000002.jpg
...,...,...,...,...,...,...,...,...,...
19145,saude202303_n_00207.json,2023-03-06 23:52:37,tweet_00163212,convo_1632891969536966663,id_1485793803495641090,VascoN95,https://twitter.com/VascoN95/status/1632891969...,Neste guia atualizado em fevereiro de 2023 sob...,saude202303_n_00207_00000001.jpg
19147,saude202303_n_00207.json,2023-03-07 09:36:34,tweet_00163219,convo_1633038922753122305,id_1501277332119511048,chris_banida,https://twitter.com/chris_banida/status/163303...,Em breve esse venen☠️ virá para as crianças. h...,saude202303_n_00207_00000002.jpg
19148,saude202303_n_00207.json,2023-03-07 08:45:13,tweet_00163225,convo_1633026002245890049,id_2370633043,unknown,unknown,Aumenta a preocupação da OMS com a gripe aviár...,saude202303_n_00207_00000003.jpg
19149,saude202303_n_00207.json,2023-03-07 02:30:35,tweet_00163233,convo_1632931722000990210,id_1602447377771773953,anamanerolemos,https://twitter.com/anamanerolemos/status/1632...,A OMS muda seu plano de pandemia (que não menc...,saude202303_n_00207_00000004.png


## Export tweets dataset

In [8]:
from pathlib import Path

output_dir = Path("corpus")
output_dir.mkdir(parents=True, exist_ok=True)

df_tweets.to_json(output_dir / "tweets.ndjson", orient="records", lines=True, force_ascii=False)
df_tweets.to_csv(output_dir / "tweets.tsv", sep="\t", index=False)
df_tweets.to_excel(output_dir / "tweets.xlsx", index=False)

print(f"Exported {len(df_tweets):,} rows to:")
print(f"- {output_dir / 'tweets.ndjson'}")
print(f"- {output_dir / 'tweets.tsv'}")
print(f"- {output_dir / 'tweets.xlsx'}")

Exported 15,787 rows to:
- corpus/tweets.ndjson
- corpus/tweets.tsv
- corpus/tweets.xlsx


## Identify empty, corrupted and duplicated images

In [1]:
import os
import hashlib
import pandas as pd
from PIL import Image

def get_image_details(directory):
    data = []

    # We use a dictionary to track hashes for duplicate detection
    # Key: MD5 hash, Value: List of file paths with that hash
    hashes = {}

    for root, _, files in os.walk(directory):
        for file in files:
            if not (file.lower().endswith(('.jpg', '.jpeg', '.png'))):
                continue

            file_path = os.path.join(root, file)
            file_size = os.path.getsize(file_path)

            is_empty = file_size == 0
            is_corrupted = False
            md5_hash = None

            if not is_empty:
                # 1. Calculate Hash for Duplicate Detection
                try:
                    with open(file_path, "rb") as f:
                        file_content = f.read()
                        md5_hash = hashlib.md5(file_content).hexdigest()
                except Exception:
                    md5_hash = "error_reading_hash"

                # 2. Check for Corruption using Pillow
                try:
                    with Image.open(file_path) as img:
                        img.verify()  # Verify integrity
                except (IOError, SyntaxError, Image.UnidentifiedImageError):
                    is_corrupted = True

            data.append({
                'filename': file,
                'path': file_path,
                'extension': os.path.splitext(file)[1].lower(),
                'size_bytes': file_size,
                'is_empty': is_empty,
                'is_corrupted': is_corrupted,
                'md5_hash': md5_hash
            })

    df = pd.DataFrame(data)

    # 3. Mark Duplicates
    # A file is a duplicate if its hash has appeared before in the dataset
    # (Excluding empty/corrupted files from hash-based duplicate logic if preferred)
    df['is_duplicate'] = df.duplicated(subset=['md5_hash'], keep='first') & df['md5_hash'].notnull()

    return df

# Usage
image_dir = 'corpus/cl_st1_ph2_tatiana_images'
df_images = get_image_details(image_dir)

# Quick summary of the findings
print(f"Total files processed: {len(df_images)}")
print(f"Empty files: {df_images['is_empty'].sum()}")
print(f"Corrupted files: {df_images['is_corrupted'].sum()}")
print(f"Duplicate files: {df_images['is_duplicate'].sum()}")

# Displaying the first few rows
df_images

Total files processed: 14450
Empty files: 239
Corrupted files: 0
Duplicate files: 1228


,filename,path,extension,size_bytes,is_empty,is_corrupted,md5_hash,is_duplicate
0,saude202201_n_00044_00000002.jpg,corpus/cl_st1_ph2_tatiana_images/saude202201_n...,.jpg,63000,False,False,f7c41166ecc6c78632b66e554921fa71,False
1,saude202303_n_00139_00000001.png,corpus/cl_st1_ph2_tatiana_images/saude202303_n...,.png,341597,False,False,827fb416bdddb19a9cc825bbe3b484f3,False
2,saude202006_n_00124_00000007.jpg,corpus/cl_st1_ph2_tatiana_images/saude202006_n...,.jpg,59966,False,False,74ab2a01cf8c56780ef343ed3faabdd6,False
3,saude202210_n_00059_00000001.jpg,corpus/cl_st1_ph2_tatiana_images/saude202210_n...,.jpg,129813,False,False,0062e72cd2e7e39a8f5faf4858b667bc,False
4,saude202007_n_00086_00000002.jpg,corpus/cl_st1_ph2_tatiana_images/saude202007_n...,.jpg,42193,False,False,dc661bb51e4e8eba1abad3495d4d3a29,False
...,...,...,...,...,...,...,...,...
14445,saude202104_n_00011_00000002.png,corpus/cl_st1_ph2_tatiana_images/saude202104_n...,.png,259044,False,False,fc3061500177f62cf3133cf8674ce7fb,False
14446,saude202103_n_00184_00000002.png,corpus/cl_st1_ph2_tatiana_images/saude202103_n...,.png,162654,False,False,dc7657fa806ad63403527f80e3c349f3,False
14447,saude202102_n_00110_00000002.jpg,corpus/cl_st1_ph2_tatiana_images/saude202102_n...,.jpg,160974,False,False,ab718e62f4c9e0139835ca55d753d488,False
14448,saude202109_n_00039_00000007.jpg,corpus/cl_st1_ph2_tatiana_images/saude202109_n...,.jpg,93154,False,False,6ab191f3f2441b475fba34d1dda98949,False


## Copy deduplicated images to a separate directory

In [2]:
import os
import shutil

# 1. Define paths
output_dir = 'corpus/deduplicated_1'

# 2. Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# 3. Filter the DataFrame for valid files only
# Criteria: Not empty, Not corrupted, and Not a duplicate
df_valid = df_images[
    (df_images['is_empty'] == False) &
    (df_images['is_corrupted'] == False) &
    (df_images['is_duplicate'] == False)
    ]

print(f"Copying {len(df_valid)} valid files to {output_dir}...")

# 4. Copy the files
for _, row in df_valid.iterrows():
    src = row['path']
    dst = os.path.join(output_dir, row['filename'])

    try:
        shutil.copy2(src, dst)
    except Exception as e:
        print(f"Error copying {row['filename']}: {e}")

print("Task complete! Your deduplicated corpus is ready.")

Copying 12983 valid files to corpus/deduplicated_1...
Task complete! Your deduplicated corpus is ready.


## Export to a file

In [3]:
from pathlib import Path

corpus_dir = Path("corpus")
corpus_dir.mkdir(parents=True, exist_ok=True)

output_stem = corpus_dir / "images_deduplicated_1"

df_images.to_json(
    output_stem.with_suffix(".ndjson"),
    orient="records",
    lines=True,
    force_ascii=False,
)

df_images.to_csv(
    output_stem.with_suffix(".tsv"),
    sep="\t",
    index=False,
    encoding="utf-8",
)

df_images.to_excel(
    output_stem.with_suffix(".xlsx"),
    index=False,
)

[
    output_stem.with_suffix(".ndjson").as_posix(),
    output_stem.with_suffix(".tsv").as_posix(),
    output_stem.with_suffix(".xlsx").as_posix(),
]

['corpus/images_deduplicated_1.ndjson',
 'corpus/images_deduplicated_1.tsv',
 'corpus/images_deduplicated_1.xlsx']